In [1]:
import numpy as np
import torch
import torch.nn.functional as F
import deepxde as dde
import matplotlib.pyplot as plt
from scipy.constants import epsilon_0, e, electron_mass
from cherab.core.atomic import hydrogen
from cherab.openadas import OpenADAS
from cherab.openadas.parse.adf11 import parse_adf11

DTYPE = torch.float64
EPS   = 1e-40
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


In [2]:
def to_tensor(x, requires_grad=False):
    return torch.as_tensor(x, dtype=DTYPE, device=DEVICE).requires_grad_(requires_grad)

data = parse_adf11(hydrogen, 'acd96_h.dat')
key = list(data.keys())[0]
log_den, log_temp, log_rates = data[key][1].values()

log_temp_t  = to_tensor(log_temp)
log_rates_t = to_tensor(log_rates)

log_rates_1d = log_rates_t.mean(dim=0)

Nt = log_temp_t.size(0)

log_sigma_mean = log_rates_1d.mean()
R_ref = torch.pow(10.0, log_sigma_mean)

In [3]:
def interp_log_sigma(ly):
    ly = ly.clamp(log_temp_t[0] + 1e-12, log_temp_t[-1] - 1e-12)
    idx = (torch.searchsorted(log_temp_t, ly) - 1).clamp(0, Nt - 2)

    x0 = log_temp_t[idx]
    x1 = log_temp_t[idx + 1]

    w = (ly - x0) / (x1 - x0)

    y0 = log_rates_1d[idx]
    y1 = log_rates_1d[idx + 1]

    return (1 - w) * y0 + w * y1


def sigmav(T, r_ref):
    T = T.clamp(min=EPS)
    ly = torch.log10(T)

    log_sigma = interp_log_sigma(ly)

    R_phys = torch.pow(10.0, log_sigma)

    return R_phys / r_ref


T_min = 10**log_temp_t[0]
T_max = 10**log_temp_t[-1]

T_test = torch.linspace(T_min * 1.01, T_max * 0.99, 200, device=DEVICE, dtype=DTYPE, requires_grad=True)

R_scaled = sigmav(T_test, R_ref)

loss = R_scaled.mean()
loss.backward()

print("R_scaled mean:", R_scaled.mean().item())
print("R_scaled std :", R_scaled.std().item())
print("Grad mean:", T_test.grad.abs().mean().item())
print("Grad max :", T_test.grad.abs().max().item())
print("Finite R:", torch.isfinite(R_scaled).all().item())
print("Finite grad:", torch.isfinite(T_test.grad).all().item())
print("logT range:", torch.log10(T_test).min().item(), torch.log10(T_test).max().item())

R_scaled mean: 5.983244188969368
R_scaled std : 84.31209812645261
Grad mean: 0.3137984985495985
Grad max : 62.7595187356277
Finite R: True
Finite grad: True
logT range: -0.6946486262173573 3.99563519459755


In [11]:
lambda_n, lambda_T = 1.0, 1.0

def ode_system(rho, y, n_max = 2.5e19, T_max = 7.0, B = 1.0):
    D0 = (2 * np.sqrt(2 * np.pi * electron_mass)/ 3) * (e / (4 * np.pi * epsilon_0 * B))**2 * n_max / np.sqrt(T_max*e) 
    kappa0 = 4.7 * n_max * D0

    def D_perp(n, T):
        b = 2 * np.log(4 * np.pi * np.power(epsilon_0 * T_max * e, 1.5) / (e**3 * np.sqrt(n_max)))
        return n / torch.sqrt(T) * ( torch.log(torch.pow(T + 1e-12, 3) / n) + b )

    n = y[:, 0:1]
    T = y[:, 1:2]

    dn_rho = dde.grad.jacobian(y, rho, i=0)
    dT_rho = dde.grad.jacobian(y, rho, i=1)

    D_perp_eval = D_perp(n, T)
    func1 = rho * D_perp_eval * dn_rho
    func2 = rho * n * D_perp_eval * dT_rho
    
    df1_rho = dde.grad.jacobian(func1, rho)
    df2_rho = dde.grad.jacobian(func2, rho)

    return [df1_rho, df2_rho + (15/47) * (rho* dn_rho * dT_rho + T * df1_rho)]


9.88183870646689e-16

In [5]:
def boundary_l(x, on_boundary):
    return on_boundary and dde.utils.isclose(x[0], 0)

def boundary_r(x, on_boundary):
    return on_boundary and dde.utils.isclose(x[0], 1)

geom = dde.geometry.Interval(0.0, 1.0)

bcDirichlet_l_n = dde.icbc.DirichletBC(geom, lambda x: 1, boundary_l, component=0)
bcDirichlet_l_T = dde.icbc.DirichletBC(geom, lambda x: 1, boundary_l, component=1)

bcNeumann_l_n = dde.icbc.NeumannBC(geom, lambda x: 0, boundary_l, component=0)
bcNeumann_l_T = dde.icbc.NeumannBC(geom, lambda x: 0, boundary_l, component=1)

bcRobin_r_n = dde.icbc.RobinBC(geom, lambda X, y: -y / lambda_n, boundary_r, component=0)
bcRobin_r_T = dde.icbc.RobinBC(geom, lambda X, y: -y / lambda_T, boundary_r, component=1)

boundaries = [bcDirichlet_l_n, bcDirichlet_l_T, bcNeumann_l_n, bcNeumann_l_T, bcRobin_r_n, bcRobin_r_T]

data = dde.data.PDE(geom, ode_system, boundaries, 300, 2, num_test=300)

layer_size = [1] + [64] * 6 + [2]
activation = "tanh"
initializer = "Glorot normal"
net = dde.nn.FNN(layer_size, activation, initializer)
model = dde.Model(data, net)
model.compile(
    "adam", lr=0.001, loss_weights=[1]*6
)

losshistory, train_state = model.train(iterations=100)

Compiling model...
'compile' took 0.000164 s

Training model...

Step      Train loss                                                      Test loss                                                       Test metric
0         [1.00e+00, 1.00e+00, 1.08e-01, 3.52e-03, 1.48e-01, 4.02e-02]    [1.00e+00, 1.00e+00, 1.08e-01, 3.52e-03, 1.48e-01, 4.02e-02]    []  
100       [1.13e-08, 2.25e-05, 7.92e-08, 2.42e-06, 5.06e-06, 3.86e-06]    [1.13e-08, 2.25e-05, 7.92e-08, 2.42e-06, 5.06e-06, 3.86e-06]    []  

Best model at step 100:
  train loss: 3.39e-05
  test loss: 3.39e-05
  test metric: []

'train' took 1.078512 s



In [6]:
print(train_state.loss_train)
print(train_state.loss_test)

[1.1256418e-08 2.2489639e-05 7.9199090e-08 2.4204223e-06 5.0608496e-06
 3.8615158e-06]
[1.1256418e-08 2.2489639e-05 7.9199090e-08 2.4204223e-06 5.0608496e-06
 3.8615158e-06]
